# Lesson 23 · Finding the steepest way up with your eyes closed — gradients: partial derivatives (∂f/∂x) assembled into a direction vector

**Learning goals**: take the partial derivative of a multivariable function with respect to each variable, and assemble them into the gradient $\nabla f$ — which points in the direction of fastest ascent.

> **"Uphill" in the title vs "downhill" in training**: it's the same arrow — uphill is $+\nabla f$, downhill is $-\nabla f$. This lesson teaches you to read it; L25 walks in the negative direction.

**Why it matters for Aurora**: `backward` computes the partial derivative of the loss with respect to every weight and assembles them into a gradient; the optimizer uses it to update the parameters.


← **Previous lesson** · [L22 · Derivatives](L22_derivatives.ipynb)

> Last lesson covered **derivatives**: tangent slope, the limit definition, numerical vs analytical differentiation.
> This lesson explores **gradients**.

## This lesson's story: where is the steepest way down the loss function?

Imagine you're standing on a mountain with your eyes closed, trying to find the fastest way down.
Your feet can feel "the slope in each direction" — the gradient packs those slopes into a single vector.

For a multivariate function f(x₁, x₂, ..., xₙ), the gradient is:

$$\nabla f = \left(\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \ldots, \frac{\partial f}{\partial x_n}\right)$$

Each component ∂f/∂xᵢ tells you: **hold every other variable fixed, move a tiny bit along xᵢ only — how does the function value change?**

The gradient vector points in the direction in which **the function value increases fastest** (steepest ascent).
To make the loss go down, walk in the **opposite direction** to the gradient — that is the whole logic of gradient descent (L25).

This lesson computes each component numerically with a central difference:

```python
∇f[i] ≈ (f(point + hᵢ) - f(point - hᵢ)) / (2h)
```

You'll implement `gradient(f, point, h=1e-5)`, which works for any loss function in Aurora.

### First, meet two new symbols: ∂ and ∇

Two symbols we didn't see in L22 just popped up in the formula above. They look a bit unfamiliar, so let's get their names and how to say them out of the way first — the symbols themselves aren't hard; what's hard is that nobody tells you what they're called.

**∂ — read as "partial" (or "partial d")**

In L22, the derivative of a single-variable function was written `f'(x)` or `df/dx`, using an ordinary letter `d`. Here it's been swapped for `∂` (it looks like a d, but it's a separate symbol you can't type on a keyboard; mathematicians call it "partial"). Why switch symbols instead of just keeping `d`?

Because `d` carries a hidden assumption: "only one variable is moving." But in a multivariable function like `f(x, y)`, `x` and `y` can change at the same time, and "how the function value changes" depends on how the two move together — that's the territory of the "total derivative", which this course won't go into. Writing `∂f/∂x` with a `∂` is a reminder: **only x is allowed to move here; y is temporarily welded in place** — a "partial" derivative, hence the name. Whenever you see `∂`, you know there are other variables behind it being held still.

**∇ — read as "nabla" or "del"**

The upside-down triangle in `∇f` is called "nabla", because its shape resembles an ancient harp called a nabla. You don't need to remember that bit of history trivia; all you need to know is: **`∇` is not a new kind of operation — it's just shorthand for the act of "lining up all the partial derivatives, in order, into one vector"**.

$$\nabla f = \left(\frac{\partial f}{\partial x_1}, \ldots, \frac{\partial f}{\partial x_n}\right)$$

Read it as "nabla f", or simply "grad f". When you see `∇f`, translate it in your head into: "take the partial derivative with respect to each variable separately, then line the results up in order as a vector." That's all there is to it.

## 1. Partial derivatives: ask about one direction at a time

**Analogy**: you're a blindfolded cook who wants to know "how fast the taste of the soup changes as you add salt."
You hold every other ingredient fixed (amount of water, temperature, heat) and change only the salt — that's a partial derivative:

$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x+h, y) - f(x-h, y)}{2h}$$

**Worked example**: f(x, y) = x² + y²

- ∂f/∂x: treat y as a constant and differentiate with respect to x → ∂f/∂x = 2x
- ∂f/∂y: treat x as a constant and differentiate with respect to y → ∂f/∂y = 2y

At the point (3, 4): ∇f(3,4) = [2×3, 2×4] = [6, 8]

**What is the direction of the gradient?**
The gradient vector [6, 8] is perpendicular to the tangent of the contour line f(x,y)=25 at (3,4), and points in the direction in which the function value increases fastest.

> When you implement `gradient(f, point)`: for each component i, build a perturbation vector of ±h and take a central difference.

### Weld y in place, then differentiate — is that really allowed?

The step above went by quickly: "treat y as a constant and differentiate with respect to x → ∂f/∂x = 2x". But why does "locking one variable" let us apply the single-variable rules from L22 directly? It isn't a shortcut; there's a rigorous reason.

**Analogy**: picture f(x, y) as a topographic map, where `x` is longitude, `y` is latitude, and `f` is elevation. Now you're standing on the line of latitude `y=4`, and you ask: "if I walk only along the longitude direction, how does the elevation change?" — that's the same as slicing out of the whole map the single profile where latitude is always 4. Along that profile, elevation varies only with longitude `x`, so it becomes a genuine **single-variable curve**.

**Written out formally**: fix `y = y₀` and define a new single-variable function

$$g(x) = f(x, y_0)$$

`g` is the profile curve left over once "y is welded at y₀". It has only one input, `x` — there's no essential difference between it and `f(x)=x²` from L22. Differentiate `g` directly using L22's definition:

$$g'(x_0) = \lim_{h \to 0} \frac{g(x_0+h) - g(x_0)}{h} = \lim_{h \to 0} \frac{f(x_0+h, y_0) - f(x_0, y_0)}{h}$$

And this limit is exactly the **definition** of the partial derivative `∂f/∂x` at `(x₀, y₀)`. In other words:

$$\frac{\partial f}{\partial x}(x_0, y_0) = g'(x_0)$$

**A partial derivative isn't a new concept at all — it's just the name for the operation "weld the other variables in place, collapse to a single-variable function, then apply L22's ordinary differentiation rules."** That's why "treating y as a constant" is legitimate: once it's welded, `y` is no longer a variable, and what's left really is an ordinary function of `x` alone.

### Deriving it step by step: where exactly does ∂f/∂x = 2x come from?

You shouldn't remember a result just by "plugging into a formula", so below we derive `∂f/∂x` for `f(x,y)=x²+y²` in full, using the central-difference definition given in this lesson — every step written out, nothing skipped.

Step 1: copy down the definition, with `y` welded in place:

$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x+h, y) - f(x-h, y)}{2h}$$

Step 2: substitute `f(x,y)=x²+y²` and expand the two terms of the numerator separately:

$$f(x+h, y) = (x+h)^2 + y^2 = x^2 + 2xh + h^2 + y^2$$
$$f(x-h, y) = (x-h)^2 + y^2 = x^2 - 2xh + h^2 + y^2$$

Step 3: subtract the two — notice that the terms whose sign doesn't change, `x²`, `h²` and `y²`, cancel outright, leaving only the term with `h` to the first power, which has opposite signs:

$$f(x+h,y) - f(x-h,y) = (2xh) - (-2xh) = 4xh$$

Step 4: divide by `2h`:

$$\frac{4xh}{2h} = 2x$$

The `h` cancels completely here, just like in L22's `f(x)=x²` example — you don't even need to actually take the `h→0` limit, because the result doesn't depend on `h` in the first place; it's simply `2x`. That is the complete origin of `∂f/∂x=2x`: not "a pattern you spot", but the result of honestly expanding, subtracting and simplifying the definition. `∂f/∂y=2y` follows from exactly the same steps (this time weld `x` and expand in `y`) — work it through yourself as an exercise; you'll get a much better feel for it than by watching someone else do it.

### Quick review: why do partial derivatives also use the "divide by 2h" central difference?

This lesson's formula reuses `(f(x+h)-f(x-h))/(2h)` directly. If you've forgotten what that is and why we divide by `2h`, it's time to flip back to [L22 · Derivatives](L22_derivatives.ipynb) — that lesson derives it in full with a Taylor expansion. Here we just do a quick review tied to this lesson's setting, without repeating the derivation.

**Three kinds of difference — recall the definitions**:

| Name | Formula | Error order |
|---|---|---|
| Forward difference | `(f(x+h)-f(x))/h` | O(h) |
| Backward difference | `(f(x)-f(x-h))/h` | O(h) |
| Central difference | `(f(x+h)-f(x-h))/(2h)` | O(h²) |

The `2` in `2h` comes from "going from `x-h` to `x+h` covers a horizontal distance of `2h`" — the numerator is the "difference in height" between the two points, the denominator is the "horizontal distance" between them, and slope = rise / run, exactly the plain idea that "two points determine a secant line". And the reason the central difference is more accurate than forward/backward (the error drops from O(h) to O(h²)) is, at root, that it's symmetric — when you Taylor-expand, subtracting the expansions at `x+h` and `x-h` cancels the largest error term (the one with `f''(x)h²`), leaving only an error of the smaller `f'''(x)h³` order. L22 checked this with concrete numbers: at `h=1e-5`, the forward-difference error is about `1e-5` and the central-difference error about `1e-10` — five orders of magnitude apart.

**Carried over to partial derivatives**, the logic is exactly the same: `∂f/∂x` is, in essence, "the derivative of the single-variable function g(x) obtained by welding y in place" (we just derived that above), so every property of differentiating g with a central difference carries over unchanged — including the rule of thumb that "the error is smallest around h≈1e-5". Let's reproduce it in code: at `(3,4)`, compare the `∂f/∂x` from a "forward difference" and from a "central difference", and see which gets closer to the analytical value `2x=6`.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

f = lambda p: p[0]**2 + p[1]**2
p = np.array([3.0, 4.0])
true_dfdx = 2 * p[0]  # analytical value = 6

print(f"{'h':>10} {'forward error':>16} {'central error':>16}")
h = 1e-1
for _ in range(6):
    forward = (f(p + [h, 0]) - f(p)) / h
    central = (f(p + [h, 0]) - f(p - [h, 0])) / (2 * h)
    print(f"{h:10.0e} {abs(forward - true_dfdx):16.3e} {abs(central - true_dfdx):16.3e}")
    h /= 10

print("\nSame conclusion as L22: the central-difference error shrinks with h much faster than the forward-difference error —")
print("which is why this lesson's partial-derivative formula uses the central difference and divides by 2h.")

### What exactly is a contour line?

The text says the gradient is "perpendicular to the contour line", but if you've never met that term, the sentence is empty. Let's start from the most intuitive place — **contour lines** come straight from maps: the loops upon loops of curves on a hiking map, where every point on one line has the same elevation, and where the lines bunch together the slope is steep. Carry that idea over into mathematics:

For a function of two variables `f(x,y)`, pick a constant `c`; join up all the points `(x,y)` that satisfy

$$f(x,y) = c$$

and you get a "contour line" (also called a level curve or isoline) — at every point on the line, the function value is the same `c`, the "height" is equal everywhere, which is also where the name "level curve" comes from.

**Why are the contour lines of `f(x,y)=x²+y²` concentric circles?** You don't memorize this; you can see it straight from the algebra: set `f(x,y)=c`, i.e. `x²+y²=c`. And `√(x²+y²)` is exactly the distance from the point `(x,y)` to the origin — so `x²+y²=c` is equivalent to "the distance to the origin is always `√c`", which is precisely the definition of a **circle** (the set of points at equal distance from a fixed point). Different values of `c` give rings of circles with different radii but the same center (all centered on the origin) — that's where the "concentric circles" come from.

Let's draw it — one look is far more intuitive than a description in words:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xg = np.linspace(-6, 6, 300)
yg = np.linspace(-6, 6, 300)
Xg, Yg = np.meshgrid(xg, yg)
Zg = Xg**2 + Yg**2  # f(x,y) = x^2 + y^2

point = np.array([3.0, 4.0])
grad_here = np.array([2 * point[0], 2 * point[1]])          # analytical gradient [6, 8]
tangent_here = np.array([-point[1], point[0]])               # tangent direction of the contour (-4, 3)

fig, ax = plt.subplots(figsize=(5, 5))
contours = ax.contour(Xg, Yg, Zg, levels=[4, 9, 16, 25, 36], colors="steelblue")
ax.clabel(contours, inline=True, fontsize=8, fmt="f=%d")

ax.quiver(*point, *grad_here, color="red", angles="xy", scale_units="xy", scale=1,
          label="gradient ∇f (perpendicular to the contour, pointing outward)")
ax.quiver(*point, *tangent_here, color="green", angles="xy", scale_units="xy", scale=1,
          label="contour tangent direction (walk along the circle, f stays the same)")
ax.scatter(*point, color=plt.rcParams["text.color"], zorder=5)
ax.annotate("(3,4)", point + [0.2, 0.2])

ax.set_xlim(-2, 10); ax.set_ylim(-2, 10)
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Contours of f(x,y)=x²+y² (concentric circles) and the gradient direction")
plt.tight_layout(); plt.show()

print("The red arrow (gradient) is perpendicular to the blue contour circle; the green arrow (tangent) runs along the circle, where f does not change.")

### How do we get the tangent direction (-4,3)? — the rotate-by-90° trick

The green tangent direction in the plot is `(-4, 3)`. It looks like numbers out of thin air, but behind it is a very general little trick that needs no memorizing: **to rotate a 2D vector by 90°, just "swap the two components and put a minus sign on one of them"**.

Concretely, any vector `(a, b)` rotated by 90° becomes `(-b, a)` (counterclockwise). Let's check that it really is perpendicular — using the dot product, since the mathematical definition of "perpendicular" is a dot product of zero:

$$(a, b) \cdot (-b, a) = a\times(-b) + b \times a = -ab + ab = 0$$

Note that this equation holds for **any** `a, b` — `-ab+ab` always cancels to zero, whatever `a` and `b` are. It's a pure algebraic identity, not a coincidence, and you don't need to re-derive it for each point.

Back to the contour lines: at the point `(3,4)`, the gradient is `(6,8)` (proportional to `(3,4)` itself, because `∇f=(2x,2y)=2(x,y)`). The tangent to a circle at a point is, geometrically, perpendicular to the radius from the center to that point — and the radial direction happens to be parallel to the gradient (both point along `(x,y)`). So the tangent direction = the radial direction rotated by 90° = `(3,4)` rotated by 90° = `(-4,3)`. In the same way, at the point `(1,2)` the radial direction is `(1,2)`, so the tangent direction is `(-2,1)` — the same "swap the components, add a minus sign" rotation trick. These aren't numbers to learn by rote; you can work them out on the spot for any point.

**This also explains a comment in the parameter-experiment code further down**: "the dot product with `tangent=[-g[1],g[0]]` is always `-ab+ba=0` (an algebraic identity, independent of the implementation, so it can't check correctness)" — because whether or not your `gradient()` function computes correctly, `(a,b)` and `(-b,a)` always have a dot product of zero, so that check would always pass and could never catch a bug. That's why the notebook code instead checks real correctness with "the gradient direction should match the direction of the position vector (radially outward)", rather than with the tautology "gradient ⊥ tangent".

### Why does "perpendicular to the contour line" mean "steepest"?

This is the point in the lesson that's easiest to gloss over, and also the one most worth explaining properly. All three students asked about the same puzzle: if you walk along a contour line, of course the function value doesn't change (that's obvious); but why is the "perpendicular" direction precisely the fastest-changing one among **all** directions, and not just "some direction where it changes"?

**The key tool: the directional derivative**. It answers the question "starting from a point, if you take a tiny step along any chosen direction `u` (a unit vector, length 1), how fast does the function value change?" The result (this course states the result here and leaves the proof to a more advanced course) is:

$$D_u f = \nabla f \cdot u = \|\nabla f\| \, \|u\| \cos\theta = \|\nabla f\| \cos\theta$$

where `θ` is the angle between the gradient vector `∇f` and your chosen direction `u` (`‖u‖=1`, so it drops out). What the formula says: **the rate of change along direction `u` equals the length of the gradient times an "alignment" factor `cosθ`** — the more closely `u` lines up with the gradient (the smaller `θ`), the larger the rate of change.

Treat the formula as a function of `θ`: when is `cosθ` largest? At `θ=0`, that is, when `u` points **exactly** along the gradient, `cosθ=1` and the rate reaches its maximum, `‖∇f‖` — that is the real source of "gradient direction = direction of steepest ascent". It isn't an intuitive guess; it comes from treating the directional derivative as a function of `θ` and maximizing it.

Now look at `θ=90°` (i.e. `u` perpendicular to the gradient): `cosθ=0`, and the directional derivative `D_u f = 0` — **walking in this direction, the instantaneous rate of change of the function value is zero**. And "the function value doesn't change" is exactly the definition of a contour line! So: **the direction perpendicular to the gradient has zero rate of change, i.e. it runs along the contour line; conversely, the tangent direction of a contour line must be perpendicular to the gradient** — those two sentences say the same thing from different angles. The gradient direction gets "squeezed" into the one remaining direction with the greatest rate of change, because it's the direction that "leaves the contour line fastest" — which is also why the red arrow (gradient) in the plot looks like it's "charging outward" from the circle rather than hugging it.

### While we're here, a name to know: what is a "normal vector"?

We've kept saying "the gradient is perpendicular to the tangent of the contour line". There's actually a shorter way to say that — in mathematics, **at a point on a curve (or surface), a vector perpendicular to the tangent direction at that point is called the curve's "normal vector" at that point**. "Normal" here means "perpendicular, orthogonal" (the same sense as a "normal line"). It isn't a new computation, just a name for the idea of "the perpendicular direction".

A circle is the most intuitive example: at any point on a circle, the tangent is the direction that "hugs the circle", and the normal vector is the direction "perpendicular to the tangent" — that is, the direction **along the radius** (either from the center toward the point, or the reverse). Draw a circle and you can confirm it at a glance: the line from the center to a point on the circle (the radius) is naturally perpendicular to the tangent at that point; that follows directly from the geometric definition of a circle.

So these three phrases all describe the same thing from different angles: **gradient direction = direction of fastest increase of the function = normal vector of the contour line at that point**. For concentric circles centered on the origin, like those of `f(x,y)=x²+y²`, the normal vector is exactly the radial direction — which is also why, when we computed the gradient `(6,8)` earlier, it turned out to point the same way as (be proportional to) the position vector `(3,4)`: for these particular circular contour lines, "normal vector" and "radial direction" are one and the same.

## Into the lab: watching a function through numerical change

Look at the values of `dfdx` and `dfdy` at the point `(3, 4)`, and confirm that the central-difference results match the analytical formulas `∂f/∂x = 2x = 6` and `∂f/∂y = 2y = 8` to within 1e-9.


In [ ]:
import numpy as np
f = lambda p: p[0]**2 + p[1]**2     # f(x,y) = x^2 + y^2
h = 1e-5
p = np.array([3.0, 4.0])
dfdx = (f(p+[h,0]) - f(p-[h,0]))/(2*h)
dfdy = (f(p+[0,h]) - f(p-[0,h]))/(2*h)
print('∂f/∂x ≈', round(dfdx,3), '(true value 2x=6)  ∂f/∂y ≈', round(dfdy,3), '(true value 2y=8)')

## Hands-on: a rate of change is more than a slogan

For the single-variable function `f(x) = x²`, estimate the slope at five points with a central difference, print the results next to the analytical value `f'(x) = 2x`, and get a feel for the numerical accuracy at h=1e-3.


In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('approx. slope =', np.round(slopes, 3))
print('theoretical slope 2x =', 2 * xs)


## Code experiment: walk through different positions and watch the slope change

Walk through seven points in `x ∈ [-3, 3]`, print the function value and the numerical slope at each, and confirm that the slope always stays close to the analytical derivative `f'(x) = 2x + 2`.


In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ Implement `gradient(f, point, h=1e-5)`

Take one central difference per component and return the gradient vector.

**Line of reasoning**:
1. Initialize `grad = np.zeros_like(point)`, so the output has the same shape as the input
2. For each dimension i, build a unit perturbation vector `e`: `e = np.zeros_like(point); e[i] = 1.0`, with all other components 0
3. Compute `(f(point + e*h) - f(point - e*h)) / (2 * h)` and store it in `grad[i]` (the two ways of writing it are equivalent; this lesson consistently uses the unit-vector-times-h form, to match the variable table below)
4. The key point: perturb only dimension i each time and leave the others unchanged — this is exactly the "hold the other variables fixed" step of a partial derivative

**Reference input and output**: f(x, y) = x² + y², point = [3, 4] → ∂f/∂x = 2×3 = 6, ∂f/∂y = 2×4 = 8 → gradient = [6, 8]

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


### Before you write code, fill in the variable table

Before writing `gradient`, pin down three things:
- Input: `f` (a callable), `point` (an n-dimensional NumPy array), `h` (step size, default 1e-5)
- Key step: for each dimension i, build a unit perturbation vector `e` and compute `(f(p + e*h) - f(p - e*h)) / (2h)`
- Return: a NumPy array of length n whose i-th element is the value of ∂f/∂xᵢ at `point`


### What exactly is the "unit perturbation vector" `e`, and why bother building it?

Step 2 of the line of reasoning has a sentence that's easy to skip: `e = np.zeros_like(point); e[i] = 1.0`. Two things here tend to trip people up: why this `e` is called a "unit vector", and why we don't just write out a list like `[0,...,h,...,0]` by hand.

**Where does the name "unit vector" come from?** Recall the three most basic directions in 3D space — along the x-axis, y-axis and z-axis respectively:

```
e_x = (1, 0, 0)      # 1 only in dimension 0, length = 1
e_y = (0, 1, 0)      # 1 only in dimension 1, length = 1
e_z = (0, 0, 1)      # 1 only in dimension 2, length = 1
```

These three vectors have two things in common: **each has length exactly 1** (the literal meaning of "unit"), and **each has only one nonzero component** — which is why they're also called "basis vectors": any vector in the space can be built by combining them, so they hold up the whole coordinate system like a foundation. The `e` in the code is the basis vector with a 1 in dimension `i` and 0 everywhere else — for dimension 0, `e` is `e_x`; for dimension 1 it's `e_y`; and so on, built the same way no matter how many dimensions there are.

**Why build the unit vector `e` first and then multiply by `h`, instead of writing `[0,...,h,...,0]` by hand?** This separates "direction" from "step size": `e` only says "which coordinate axis to move along" (length 1, pure direction), and multiplying by `h` then decides "how far to move" (`e*h` becomes `[0,...,h,...,0]`). Splitting it up like this has a direct benefit — **it doesn't depend on how many dimensions the input has**. `point` might be a 2D `[x,y]`, or it might be a 100-dimensional vector of model parameters; `e = np.zeros_like(point); e[i] = 1.0` inside a `for i in range(len(point))` loop works for any number of dimensions, without knowing `n` in advance or writing out `n` lists of different lengths. If you wrote literal lists by hand instead, a 100-dimensional input would need 100 different lists, and the code couldn't be reused at all — the unit-vector form is, at heart, trading "parameterize by the loop variable `i`" for generality.

### Is `dtype=float` overkill, or necessary?

The first line of the code reads `point = np.asarray(point, dtype=float)`. If the caller already passes numbers with decimal points, like `[3.0, 4.0]`, it really does look like overkill — but if someone passes `[3, 4]` (integers, no decimal points), a hard-to-spot bug appears. Let's just run it and see what goes wrong.

In [ ]:
import numpy as np

# What happens to a list of integers if we don't convert it to float?
point_int = np.asarray([3, 4])          # no dtype=float, so numpy infers an int type from the integers
print("dtype of point_int:", point_int.dtype)

grad_bug = np.zeros_like(point_int)     # np.zeros_like copies point_int's dtype — also int!
grad_bug[0] = 6.2831                    # we want to store a partial-derivative result of 6.2831...
print("Wrong: grad_bug =", grad_bug, " dtype =", grad_bug.dtype)
print("→ 6.2831 was silently truncated to the integer 6! The array's dtype did this; no error, just a quietly wrong answer.")

print()

# With dtype=float added:
point_fixed = np.asarray([3, 4], dtype=float)   # force conversion to floating point
grad_fixed = np.zeros_like(point_fixed)          # this time zeros_like inherits the float type
grad_fixed[0] = 6.2831
print("Right: grad_fixed =", grad_fixed, " dtype =", grad_fixed.dtype)
print("→ The decimals are kept intact.")

This is exactly the trap that the first line of `gradient()`, `point = np.asarray(point, dtype=float)`, guards against: inside the function, `grad = np.zeros_like(point)` copies `point`'s dtype, so without forcing a conversion to `float` up front, as soon as someone passes a list of integers (say `gradient(f, [3, 4])`), the computed partial derivatives get silently truncated to integers, and **no error is raised** — this is one of the hardest kinds of numerical bug to track down: the program runs normally and produces output, but the result is wrong. Locking down the input type with `dtype=float` at the start rules out this whole class of problems.

In [ ]:
def gradient(f, point, h=1e-5):
    point = np.asarray(point, dtype=float)
    grad = np.zeros_like(point)
    # ✏️ TODO: fill in every component of grad
    raise NotImplementedError("TODO: compute each partial derivative ∂f/∂xᵢ = (f(x+ε·eᵢ) - f(x-ε·eᵢ)) / (2ε)")

In [ ]:
try:
    g = gradient(lambda p: p[0]**2 + p[1]**2, [3.0, 4.0])
except (NotImplementedError, TypeError):
    print("⬜ Finish the gradient exercise above first, then run this cell")
else:
    print('gradient =', np.round(g, 3), '(should be ≈ [6, 8])')
    assert np.allclose(g, [6.0, 8.0], atol=1e-4)
    print('✅ Passed: you can compute a gradient.')

**🔗 Aurora connection**: during training, the loss function `L(W₁, W₂, …)` is a multivariable function of all the weights; what `backward()` computes is exactly this function's gradient, with one component of the gradient vector per weight. Aurora doesn't implement analytical backpropagation yet, so this lesson's numerical gradient is its starting point. Backpropagation uses the chain rule to compress "probe one parameter at a time" — 2n forward passes — into 1 forward pass + 1 backward pass; the more parameters there are, the bigger the speed gap between the two. The next lesson covers the chain rule and explains where that compression comes from.

## Parameter experiment: the gradient direction is the contour line's normal vector

The code below checks `gradient` at point = [3, 4]. Following the same pattern, change point to [1, 2]: calling `gradient(f, [1.0, 2.0])` should give [2, 4] (analytical values: ∂f/∂x = 2×1 = 2, ∂f/∂y = 2×2 = 4).

Check the geometric meaning of the gradient direction: the contour lines of f(x, y) = x² + y² are concentric circles centered on the origin; at the point (1, 2), the tangent direction of the contour is (-2, 1) (its dot product with the gradient (2, 4) is zero), while the gradient direction (2, 4) points away from the center, the direction in which the function value increases fastest. Negate the gradient and you get (-2, -4), pointing toward the center — exactly the direction each step of gradient descent moves in.

In [ ]:
# Parameter experiment: gradient direction = normal vector of the contour line
f_2d = lambda p: p[0]**2 + p[1]**2

point = [3.0, 4.0]
try:
    g = gradient(f_2d, point)
except (NotImplementedError, TypeError):
    print("⬜ Finish the gradient exercise above first, then run this cell")
else:
    print(f'gradient(x²+y², [3,4]) = {np.round(g, 6)}')
    print(f'analytical [2×3, 2×4]  = [6, 8]')
    assert np.allclose(g, [6.0, 8.0], atol=1e-4), f'wrong gradient: {g}'
    print('✅ Numerical gradient matches the analytical value')

    # Check the gradient direction: for f(x,y)=x²+y², the gradient at p should point the same way as p (radially outward)
    # Note: the dot product with tangent=[-g[1],g[0]] is always -ab+ba=0 (an algebraic identity), independent of the implementation, so it can't check correctness.
    # Comparing against the analytical radial direction instead really does test whether gradient() is right.
    grad_dir = g / np.linalg.norm(g)
    radial = np.array(point, dtype=float) / np.linalg.norm(np.array(point, dtype=float))
    assert np.allclose(grad_dir, radial, atol=1e-4), \
        f"wrong gradient direction: {grad_dir}, expected the radial direction: {radial}"
    print(f'✅ Gradient direction matches the analytical radial direction (grad_dir≈{np.round(grad_dir,4)}, radial={np.round(radial,4)})')


In [ ]:
# Gradient of a function of three variables: f(x,y,z) = x² + 2y² + 3z² at (1,1,1)
f_3d = lambda p: p[0]**2 + 2*p[1]**2 + 3*p[2]**2
try:
    g3 = gradient(f_3d, [1.0, 1.0, 1.0])
except (NotImplementedError, TypeError):
    print("⬜ Finish the gradient exercise above first, then run this cell")
else:
    print(f'gradient(x²+2y²+3z², [1,1,1]) = {np.round(g3, 6)}')
    print(f'analytical                     = [2, 4, 6]')
    assert np.allclose(g3, [2.0, 4.0, 6.0], atol=1e-4)
    print('✅ Three-variable gradient check passed')
    print(f'gradient norm = {np.linalg.norm(g3):.4f} (the step-size scaling factor along the steepest direction)')


### What is the gradient norm? Why call it a "step-size scaling factor"?

The code above printed a line: `gradient norm = 7.4833 (the step-size scaling factor along the steepest direction)`. The gradient itself is a vector `[2,4,6]`, with both a "direction" and a "length" — the sections so far were all about direction (where it's steepest); the "norm" here is about the length half.

**How do you compute the norm?** It's the Euclidean norm (L2 norm) from [L11](../2_linear_algebra/L11_norms.ipynb) — square each component, add them up, take the square root:

$$\|\nabla f\| = \sqrt{2^2+4^2+6^2} = \sqrt{4+16+36} = \sqrt{56} \approx 7.4833$$

It's the same idea as "the absolute value of the slope measures steepness" for a single-variable function: the bigger the slope, the steeper the curve; here, the longer the gradient, the faster the function value rises in the steepest direction the gradient points along.

**Why call it a "step-size scaling factor"? What does it have to do with gradient descent (L25)?** The gradient descent update rule is:

```
new position = old position - learning rate (lr) × gradient
```

The distance you actually move in one step = `lr × ‖∇f‖`. Notice that the learning rate `lr` is a fixed number you set by hand, but **what really decides how far a step goes also includes the length of the gradient** — where the slope is steep (`‖∇f‖` large), the same `lr` takes you a big stride; as you near the bottom and the slope flattens out (`‖∇f‖` shrinks toward 0), the same `lr` takes you only a small step. This "automatically slowing down" effect is the gradient's length dynamically "scaling" the size of each step for you — no need to adjust `lr` by hand; the gradient norm already builds in the intuition "the flatter it gets, the slower you walk". It's also why optimization algorithms naturally slow down and gradually converge as they approach the optimum (where the gradient goes to 0), instead of charging past it with steps of the same size.

## Wrap-up

You can now call `gradient(f, point)` to compute the numerical gradient of any multivariable function at any point, returning a vector of partial derivatives the same length as `point`. Each dimension of the gradient corresponds to one input variable, and it points in the direction in which `f` increases fastest; in training, flipping the sign gives the descent direction. Aurora's `backward` computes this same vector, only by analytical derivation instead of numerical perturbation, which is orders of magnitude faster.

Next lesson: **L24** (the chain rule) explains why the gradient of a composite function equals the product of each layer's derivatives.

---
⬇️ **Clearing check**: you've read the wrap-up; now do the whiteboard challenge below before ticking off the self-review.


## ✏️ Whiteboard challenge: gradients by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: f(x, y) = x² + y². Write down the analytical formulas for ∂f/∂x and ∂f/∂y.

**Question 2**: Using the formulas from Question 1, compute ∇f(3, 4) = ?

**Question 3**: f(x, y, z) = x² + 2y² + 3z². What is the gradient ∇f at the point (1, 1, 1)?

**Question 4**: What is the geometric meaning of the gradient vector's direction?
(Hint: how is it related to the contour lines?)

When you've finished the derivations, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: ∂f/∂x = 2x, ∂f/∂y = 2y
f_2d = lambda p: p[0]**2 + p[1]**2
dfdx = lambda p: 2 * p[0]
dfdy = lambda p: 2 * p[1]
p = np.array([3., 4.])
assert np.isclose(dfdx(p), 6.0) and np.isclose(dfdy(p), 8.0)
print(f"Q1 ✅  ∂f/∂x=2x, ∂f/∂y=2y  (at ({p[0]:.0f},{p[1]:.0f}) these are {dfdx(p):.0f} and {dfdy(p):.0f})")

# Q2: ∇f(3,4) = [6, 8]
grad_analytical = np.array([dfdx(p), dfdy(p)])
assert np.allclose(grad_analytical, [6., 8.], atol=1e-10)
try:
    grad_num = gradient(f_2d, p)
    assert np.allclose(grad_num, grad_analytical, atol=1e-7), \
        f"numerical gradient {grad_num} and analytical value {grad_analytical} disagree"
    print(f"Q2 ✅  ∇f(3,4)={grad_analytical}, numerical check={np.round(grad_num,6)}")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement gradient() first, then run the answer-check cell")

# Q3: ∇f(1,1,1) = [2, 4, 6] for f=x²+2y²+3z²
f_3d = lambda p: p[0]**2 + 2*p[1]**2 + 3*p[2]**2
q3 = np.array([1., 1., 1.])
grad_3d_analytical = np.array([2., 4., 6.])
try:
    grad_3d = gradient(f_3d, q3)
    assert np.allclose(grad_3d, grad_3d_analytical, atol=1e-7)
    print(f"Q3 ✅  ∇f(1,1,1)={grad_3d_analytical}, numerical={np.round(grad_3d,6)}")
except (NotImplementedError, TypeError):
    print("⬜ Q3: implement gradient() first, then run the answer-check cell")

# Q4: geometric meaning of the gradient (numerical check: gradient ⊥ contour tangent)
# At (3,4), the tangent direction of the contour x²+y²=25 is (-y, x)/||...|| = (-4,3)/5
tangent = np.array([-4., 3.]) / 5.0
grad_norm = grad_analytical / np.linalg.norm(grad_analytical)
dot = abs(np.dot(tangent, grad_norm))
assert dot < 1e-10, f"the gradient should be perpendicular to the contour tangent; dot product={dot}"
print(f"Q4 ✅  ∇f(3,4)=[6,8] is perpendicular to the contour tangent (-4,3); dot product={dot:.2e}")
print("     gradient = direction of fastest increase = normal vector of the contour line")
print("\n🎉 Gradient whiteboard challenge passed! You've internalized steepest ascent for multivariable functions.")

In [ ]:
try:
    # ✏️ Self-review for this lesson
    l23_review = {
        "partial_derivative_formula":  None,  # understand the partial-derivative definition of ∂f/∂xᵢ? True/False
        "gradient_implemented":        None,  # gradient(f, point) implemented and assertions pass? True/False
        "gradient_direction_meaning":  None,  # gradient = steepest ascent = contour normal? True/False
        "3d_gradient_computed":        None,  # can compute the gradient of a 3-variable function? True/False
        "whiteboard_passed":           None,  # whiteboard challenge done on paper? True/False
    }

    unfilled = [k for k, v in l23_review.items() if v is None]
    assert not unfilled, f'not filled in yet: {unfilled}'
    weak = [k for k, v in l23_review.items() if v is False]
    if weak:
        print(f'⚠️  needs more work: {weak}')
    else:
        print('✅ L23 fully cleared! On to L24: the chain rule')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO exercises above first, then run this cell')

---

→ **Next lesson** · [L24 · The chain rule](L24_chain_rule.ipynb)

> Next lesson covers **the chain rule**: differentiating nested functions, the maths behind backpropagation.